---
title: "Figures: Aiyagari: Precautionary Saving in GE"
execute:
  depends_on_env: [IC_EXEC_ENV]
---

Open this notebook in [Colab](https://colab.research.google.com/github/intertemporal-choice/intertemporal-choice.github.io/blob/main/content/consumption/Aiyagari/Aiyagari.ipynb) or [Binder](https://mybinder.org/v2/gh/intertemporal-choice/intertemporal-choice.github.io/main?labpath=content/consumption/Aiyagari/Aiyagari.ipynb) to run it without installing anything.

Computes the stationary equilibrium of the economy in [Aiyagari: Precautionary Saving in GE](#sec:Aiyagari) with the code of Econ-ARK's REMARK [`Aiyagari1994QJE`](https://github.com/econ-ark/Aiyagari1994QJE), which the book installs as the package `aiyagari1994qje`; draws the chapter's figures; and checks every number the chapter quotes. The first sections build the household side step by step with HARK, as the REMARK does; one call then solves for the equilibrium. Change the calibration to see how the equilibrium responds.

In [ ]:
# Run anywhere: install the book's package (it brings econ-ark, numpy and matplotlib) if it
# is missing. Inside the book's own environment this does nothing.
try:
    import intertemporal_choice
except ImportError:
    %pip install -q git+https://github.com/intertemporal-choice/intertemporal-choice.github.io

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import aiyagari1994qje as aiyagari
from intertemporal_choice import style

style.use_book_style()

## The calibration

Aiyagari's baseline: a period is a year; $\beta=0.96$, a capital share $\alpha=0.36$ and depreciation $\delta=0.08$; relative risk aversion $\rho=3$. The log of the idiosyncratic labor productivity component $\ell$ follows an AR(1) with persistence $\tau=0.6$ and unconditional standard deviation $\sigma=0.2$, approximated by a seven-state Tauchen chain. No borrowing is allowed. The REMARK simulates 1,000 households for 1,000 years and averages over the last 700.

In [ ]:
cal = aiyagari.Calibration()  # Aiyagari's baseline; pass, e.g., income_persistence=0.9 to change it
cal

## The household side, step by step

Tauchen's method turns the AR(1) into a seven-state Markov chain: a grid of values of $\log\ell$ and a matrix $P$ whose row $i$ gives the probabilities of moving from state $i$ to each state next year.

In [ ]:
x, P = aiyagari.tauchen_chain(n=cal.income_states, tau=cal.income_persistence, sigma=cal.income_standard_deviation)
print("log labor productivity in each state:", np.round(x, 2))
print("probability of staying in the same state next year:", np.round(np.diag(P), 3))  # the diagonal of P

HARK's `MarkovConsumerType` is a consumer whose income follows such a chain. The REMARK solves its problem with the wage normalized to one: with wage $\mathsf{W}$, a household with resources $m$ consumes $\mathsf{W}\,c(m/\mathsf{W})$, so the wage drops out. At a trial interest rate, the households' problem is solved and a population of them simulated; their mean assets, scaled by the wage that rate implies, are the capital they supply. Firms' demand at the same rate comes from their first-order condition.

In [ ]:
households = aiyagari.make_households(cal, (x, P))  # a HARK MarkovConsumerType with 1,000 simulated households
r_trial = 0.03
aiyagari.simulate(households, r_trial)  # solve the household's problem at r_trial, then simulate the households
L = aiyagari.labor_supply(households)  # mean labor productivity: about 1.02, since the chain is not normalized
supply = aiyagari.capital_supply(households, [r_trial], cal)[0]  # the wage at r_trial times mean assets
demand = aiyagari.capital_demand(r_trial, cal, L)
print(f"at r = {100 * r_trial:.1f}%: households supply {supply:.2f}, firms demand {demand:.2f}")

At 3 percent firms want more capital than households hold, so the interest rate must be higher. The equilibrium is the rate at which the two meet; `solve_equilibrium` searches for it, repeating these steps.

## The equilibrium

In [ ]:
eq = aiyagari.solve_equilibrium(cal)
print(f"complete markets: r = {100 * eq.time_preference_rate:.2f}%, saving rate {100 * eq.saving_rate_complete_markets:.2f}%")
print(f"uninsured risk:   r = {100 * eq.interest_rate:.2f}%, saving rate {100 * eq.saving_rate_from_capital:.2f}%, "
      f"capital {100 * (eq.capital / eq.capital_complete_markets - 1):.1f}% higher")
print(f"labor supply L = {eq.labor_supply:.2f}, wage W = {eq.wage:.2f}")

The saving rate is Aiyagari's: replacement investment over output, $\delta K/Y=\delta\alpha/(r+\delta)$.

## The consumption functions

In [ ]:
fig, ax = plt.subplots()
aiyagari.plot_consumption_functions(eq, ax)  # one consumption function per income state, at the equilibrium r
style.save(fig, "ConsFuncByIncomeState")

## Capital supply and demand

The supply curve is computed at 48 interest rates, letting the wage move with the interest rate as it does in equilibrium. As $r$ approaches the time preference rate, the assets households want to hold grow without bound. This cell takes about half a minute.

In [ ]:
fig, ax = plt.subplots()
aiyagari.plot_capital_market(eq, ax)
style.save(fig, "CapitalSupplyDemand")

## The distribution of wealth

The U.S. curve is net worth in the 2004 Survey of Consumer Finances, the data Carroll, Slacalek, Tokuoka and White (2017) used, from the copy distributed with HARK. Its Lorenz curve is committed in `data/`; the next cell recomputes it from HARK's file and checks it.

In [ ]:
from HARK.Calibration.load_data import load_SCF_wealth_weights

us = np.loadtxt("data/SCF2004-networth-lorenz.csv", delimiter=",", comments="#", skiprows=7)  # columns: households, wealth
wealth, weight = load_SCF_wealth_weights()  # net worth and survey weight of each household
order = np.argsort(wealth, kind="stable")  # poorest first
households_share = np.r_[0, np.cumsum(weight[order])] / weight.sum()
wealth_share = np.r_[0, np.cumsum((wealth * weight)[order])] / (wealth * weight).sum()
assert np.allclose(np.interp(us[:, 0], households_share, wealth_share), us[:, 1], atol=1e-6)

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 5.0))
aiyagari.plot_wealth_lorenz(eq, ax, us_data=False)  # the model's Lorenz curve and the 45-degree line
ax.plot(us[:, 0], us[:, 1], color="C3", label="U.S. net worth, SCF 2004")
ax.set_ylim(-0.05, 1)  # the poorest U.S. households have negative net worth
ax.legend(loc="upper left", fontsize="small")
style.save(fig, "WealthLorenz")

## The numbers the chapter quotes

These checks stop the build if a number the chapter quotes stops matching the computation.

In [ ]:
model_poorest = lambda q: np.interp(q, *eq.lorenz_curve())  # the poorest fraction q's share of wealth
us_poorest = lambda q: np.interp(q, us[:, 0], us[:, 1])

assert round(100 * eq.time_preference_rate, 2) == 4.17 and round(100 * eq.saving_rate_complete_markets, 2) == 23.67
assert round(100 * eq.interest_rate, 2) == 3.86 and round(100 * eq.saving_rate_from_capital, 2) == 24.28
assert round(100 * (eq.capital / eq.capital_complete_markets - 1), 1) == 4.0
assert round(eq.gini, 2) == 0.38 and round(100 * eq.share_of_richest(0.01)) == 4
assert round(100 * eq.share_at_borrowing_constraint, 1) == 0.3
assert round(eq.average_marginal_propensity_to_consume, 3) == 0.054 and round(1 - eq.return_patience_factor, 3) == 0.038
assert round(100 * model_poorest(0.4)) == 15 and round(100 * us_poorest(0.4)) == 1
assert us_poorest(0.2) < 0 and round(100 * (1 - us_poorest(0.99))) == 32
us_gini = 1 - np.sum(np.diff(households_share) * (wealth_share[1:] + wealth_share[:-1]))  # area between the curve and equality, doubled
assert round(us_gini, 2) == 0.80

## A volatile and persistent income

Aiyagari's largest effect, with $\sigma=0.4$, $\tau=0.9$ and $\rho=5$: a negative interest rate and a saving rate far above the complete-markets one.

In [ ]:
extreme = aiyagari.solve_equilibrium(aiyagari.Calibration(income_standard_deviation=0.4, income_persistence=0.9, relative_risk_aversion=5))
print(f"r = {100 * extreme.interest_rate:.2f}%, saving rate {100 * extreme.saving_rate_from_capital:.2f}%")
assert extreme.interest_rate < 0 and 0.36 < extreme.saving_rate_from_capital < 0.38